# 🎯 DGNet: Deep Gradient Network Model, Training & MLflow Tracking
### Architecture, Dedicated Notebook MLflow Tracker, COD Benchmark & Edge Deployment

This notebook provides the complete **DGNet (Deep Gradient Network)** model execution pipeline with a **dedicated, fresh MLflow tracker** built natively inside this notebook using standard `mlflow`.
*(For data ingestion, inspection, Canny gradients, and tactical augmentations, see `notebooks/preprocess.ipynb`)*.

#### Pipeline Modules:
1. **Hardware & Environment**: GPU memory growth and OpenCV thread safety.
2. **Hyperparameters & Configuration**: Centralized project parameters.
3. **Robust Dataset Ingestion**: Loads both raw (748) and augmented (3,740) samples = 4,488 training pairs.
4. **DGNet Architecture**: Multi-scale Context Encoder + Canny Texture Encoder + GIT Cross-Attention + NCD Decoder with Deep Supervision.
5. **Dedicated Notebook MLflow Tracker**: Fresh, self-contained tracker connected directly to the local SQLite database (`metrics/mlflow_db/mlflow.db`).
6. **Loss Function & Optimization Graph**: Composite Weighted BCE + Weighted IoU + Canny Edge BCE (`DGNetLoss`) with AdamW and gradient clipping.
7. **Interactive Training Loop with Live MLflow Tracking**: Batch-level loss tracking every 10 batches, epoch-level metrics, and checkpoint artifact logging.
8. **COD Benchmark Evaluation**: Evaluates test split and logs official metrics (Pixel Acc, MAE, IoU, Dice, $F_\beta$, Precision, Recall, Specificity, FPS) directly to MLflow.
9. **Clean Black-and-White Visual Comparison**: Generates pure binary masks (no color heatmaps/overlays) labeled with model names and logs them as MLflow artifacts.
10. **TFLite INT8 Edge Quantization**: Quantization to `models/quantized/dgnet_s_int8.tflite` with MLflow artifact registration.
11. **MLflow UI & Run Query**: Queries the SQLite database and displays logged runs with UI launch instructions.

## 1. Environment Setup & Hardware Configuration
Configure TensorFlow GPU memory growth, OpenCV thread contention fix, and import core model utilities.

In [31]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path

import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import mlflow
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT.resolve()}")
print(f"TensorFlow:   {tf.__version__}")
print(f"MLflow:       {mlflow.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"✓ GPU Detected: {gpu.name} (Memory growth enabled)")
        except Exception as e:
            print(f"Note: {e}")
else:
    print("ℹ No physical GPU detected; running on CPU.")

from src.models.dgnet import DGNet, DGNetLoss, build_dgnet_model
from src.utils.config import load_project_config
from src.utils.metrics import CODMetricsAccumulator, compute_all_metrics

Project Root: C:\Users\Acer\Desktop\COD_PROJECT
TensorFlow:   2.19.0
MLflow:       3.10.0
ℹ No physical GPU detected; running on CPU.


## 2. Configuration & Hyperparameters
Loads project parameters from `params.yaml` and sets up target directories.

In [32]:
cfg = load_project_config()
data_cfg = cfg.get("data", cfg.get("dataset", {}))
model_cfg = cfg.get("model", {})
train_cfg = cfg.get("train", {})

IMG_SIZE = tuple(data_cfg.get("image_size", [384, 384]))
BATCH_SIZE = train_cfg.get("batch_size", 8)
LEARNING_RATE = train_cfg.get("learning_rate", 1e-4)
EPOCHS = train_cfg.get("epochs", 100)
BACKBONE = model_cfg.get("context_backbone", "mobilenet_v3_large")
GRADIENT_KERNEL = model_cfg.get("gradient_kernel", "sobel")
MODEL_VERSION = "v2.0_official_dgnet"
EXPERIMENT_NAME = "COD-Camouflage-Detection"

DATA_DIR = PROJECT_ROOT / "data/raw/dataset-splitM"
AUGMENTED_DIR = PROJECT_ROOT / "data/augmented/military_tactical"
CHECKPOINT_DIR = PROJECT_ROOT / "models/raw_checkpoints"
METRICS_DIR = PROJECT_ROOT / "metrics"
PRED_DIR = PROJECT_ROOT / "experiments/predictions"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Model Version:   {MODEL_VERSION}")
print(f"Backbone:        {BACKBONE}")
print(f"Image Size:      {IMG_SIZE}")
print(f"Batch Size:      {BATCH_SIZE}")
print(f"Learning Rate:   {LEARNING_RATE}")
print(f"Epochs:          {EPOCHS}")

Experiment Name: COD-Camouflage-Detection
Model Version:   v2.0_official_dgnet
Backbone:        mobilenet_v3_large
Image Size:      (384, 384)
Batch Size:      8
Learning Rate:   0.01
Epochs:          1


## 3. Robust Dataset Pipeline Ingestion
Scans raw dataset splits (`Training`, `Testing`) as well as offline tactical augmented samples. Matches image stems with masks case-insensitively (`images`/`Images` and `gt`/`GT`).

In [33]:
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def find_image_mask_pairs(base_dir: Path, split_name: str):
    """Finds matched image and mask pairs supporting case-insensitive paths."""
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    split_dir = base_dir / split_name
    if not split_dir.exists():
        return [], []
        
    img_candidates = [split_dir / "images", split_dir / "Images", split_dir]
    gt_candidates = [split_dir / "GT", split_dir / "gt", split_dir / "Masks", split_dir / "masks"]
    
    img_dir = next((d for d in img_candidates if d.exists() and d.is_dir()), None)
    gt_dir = next((d for d in gt_candidates if d.exists() and d.is_dir()), None)
    
    if not img_dir or not gt_dir:
        return [], []
        
    imgs = sorted([str(p) for p in img_dir.iterdir() if p.suffix.lower() in valid_exts])
    matched_imgs, matched_masks = [], []
    for ip in imgs:
        stem = Path(ip).stem
        for ext in [".png", ".jpg", ".jpeg", ".bmp"]:
            gp = gt_dir / f"{stem}{ext}"
            if gp.exists():
                matched_imgs.append(ip)
                matched_masks.append(str(gp))
                break
    return matched_imgs, matched_masks

# Discover raw training and testing sets
raw_train_imgs, raw_train_masks = find_image_mask_pairs(DATA_DIR, "Training")
test_imgs, test_masks = find_image_mask_pairs(DATA_DIR, "Testing")

# Discover augmented training samples
aug_train_imgs, aug_train_masks = find_image_mask_pairs(AUGMENTED_DIR, "Training")

# Combine all training samples
train_imgs = raw_train_imgs + aug_train_imgs
train_masks = raw_train_masks + aug_train_masks

print(f"✓ Loaded Raw Training Samples:       {len(raw_train_imgs)}")
print(f"✓ Loaded Augmented Training Samples: {len(aug_train_imgs)}")
print(f"✓ Total Combined Training Samples:   {len(train_imgs)}")
print(f"✓ Loaded Testing Samples:            {len(test_imgs)}")

def compute_object_gradient_np(image_uint8: np.ndarray, mask_uint8: np.ndarray) -> np.ndarray:
    if image_uint8.ndim == 4:
        image_uint8 = image_uint8[0]
    if mask_uint8.ndim == 3 and mask_uint8.shape[-1] == 1:
        mask_uint8 = mask_uint8[:, :, 0]
    canny = cv2.Canny(image_uint8, 64, 128)
    fore = ((canny > 0) & (mask_uint8 >= 128)).astype(np.float32)
    return fore[:, :, np.newaxis]

def parse_sample(img_path: tf.Tensor, mask_path: tf.Tensor, image_size: tuple):
    img_bytes = tf.io.read_file(img_path)
    image_raw = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image_raw, image_size)
    image_uint8 = tf.cast(image, tf.uint8)

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask_bytes, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, image_size, method="nearest")
    mask_bin = tf.cast(mask >= 128, tf.float32)

    def _py_grad(im, m):
        return compute_object_gradient_np(im, m)

    grad_gt = tf.numpy_function(_py_grad, [image_uint8, tf.cast(mask, tf.uint8)], tf.float32)
    grad_gt.set_shape([image_size[0], image_size[1], 1])

    image_float = tf.cast(image, tf.float32) / 255.0
    image_float.set_shape([image_size[0], image_size[1], 3])
    mask_bin.set_shape([image_size[0], image_size[1], 1])

    return image_float, mask_bin, grad_gt

def create_tf_dataset(image_paths, mask_paths, image_size=(384, 384), batch_size=8, is_train=True):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    dataset = dataset.map(lambda x, y: parse_sample(x, y, image_size), num_parallel_calls=tf.data.AUTOTUNE)
    if is_train:
        dataset = dataset.shuffle(buffer_size=min(len(image_paths), 256), seed=42)
        dataset = dataset.map(lambda img, m, g: ((img - IMAGENET_MEAN) / IMAGENET_STD, m, g), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = dataset.map(lambda img, m, g: ((img - IMAGENET_MEAN) / IMAGENET_STD, m, g), num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

✓ Loaded Raw Training Samples:       748
✓ Loaded Augmented Training Samples: 3740
✓ Total Combined Training Samples:   4488
✓ Loaded Testing Samples:            330


## 4. Build DGNet Architecture
Instantiate the dual-branch DGNet model and inspect the parameter breakdown.

In [34]:
model = build_dgnet_model(
    backbone_name=BACKBONE,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    gradient_kernel=GRADIENT_KERNEL,
    pretrained=True
)

dummy_input = tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32)
_ = model(dummy_input, training=False)

total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print("=" * 65)
print(f" DGNet Architecture: {BACKBONE} + {GRADIENT_KERNEL}")
print("=" * 65)
print(f" Total Parameters:      {total_params:,} ({total_params*4 / (1024*1024):.2f} MB)")
print(f" Trainable Parameters:  {trainable_params:,}")
print("=" * 65)

 DGNet Architecture: mobilenet_v3_large + sobel
 Total Parameters:      3,578,442 (13.65 MB)
 Trainable Parameters:  3,553,850


## 5. Dedicated Notebook MLflow Tracker (Fresh & Self-Contained)
A clean, fresh MLflow tracking engine written natively inside this notebook using standard `mlflow`.
- Connects directly to the project's local SQLite store: `sqlite:///metrics/mlflow_db/mlflow.db`
- Sets the active experiment to `COD-Camouflage-Detection`
- Logs parameters, configuration YAML files, architecture summaries, real-time batch metrics, and artifacts.

In [35]:
class DGNetNotebookMLflowTracker:
    """
    Self-contained MLflow experiment tracker built specifically for this notebook.
    Uses native mlflow API to persist runs directly to metrics/mlflow_db/mlflow.db.
    """
    def __init__(self, experiment_name: str = "COD-Camouflage-Detection"):
        self.experiment_name = experiment_name
        self.active_run = None
        
        # Configure SQLite database path
        db_dir = (PROJECT_ROOT / "metrics" / "mlflow_db").resolve()
        db_dir.mkdir(parents=True, exist_ok=True)
        db_path = db_dir / "mlflow.db"
        self.tracking_uri = f"sqlite:///{db_path.as_posix()}"
        
        mlflow.set_tracking_uri(self.tracking_uri)
        mlflow.set_experiment(self.experiment_name)
        print(f"[MLflow] Connected to database: {self.tracking_uri}")
        print(f"[MLflow] Active Experiment:     '{self.experiment_name}'")
        
    def start_run(self, run_name: str = None, tags: dict = None, description: str = None):
        """Starts a new MLflow run."""
        default_tags = {
            "project": "COD",
            "task": "camouflage_detection",
            "framework": "tensorflow_2.x",
            "notebook": "DGnet.ipynb"
        }
        if tags:
            default_tags.update(tags)
            
        self.active_run = mlflow.start_run(
            run_name=run_name,
            tags=default_tags,
            description=description
        )
        print(f"[MLflow] ✓ Started Run ID: {self.active_run.info.run_id} (Name: '{run_name}')")
        return self.active_run

    def log_params(self, params: dict):
        """Logs hyperparameters to active run."""
        if not self.active_run:
            return
        clean_params = {k: (str(v) if isinstance(v, (list, tuple)) else v) for k, v in params.items()}
        mlflow.log_params(clean_params)

    def log_metrics(self, metrics: dict, step: int = None):
        """Logs scalar metrics at an optional step or epoch."""
        if not self.active_run:
            return
        clean_metrics = {k: float(v) for k, v in metrics.items()}
        mlflow.log_metrics(clean_metrics, step=step)

    def log_artifact(self, local_path: str, artifact_path: str = None):
        """Logs a file or directory as an MLflow artifact."""
        if not self.active_run:
            return
        p = Path(local_path)
        if p.exists():
            if p.is_dir():
                mlflow.log_artifacts(str(p), artifact_path=artifact_path)
            else:
                mlflow.log_artifact(str(p), artifact_path=artifact_path)

    def log_configs(self, config_paths: list, artifact_path: str = "configs"):
        """Logs project YAML configs for experiment reproducibility."""
        for cp in config_paths:
            self.log_artifact(str(cp), artifact_path=artifact_path)

    def log_model_summary(self, model_obj):
        """Logs text model summary artifact."""
        if not self.active_run:
            return
        summary_path = PROJECT_ROOT / "metrics" / "notebook_model_summary.txt"
        with open(summary_path, "w", encoding="utf-8") as f:
            model_obj.summary(print_fn=lambda x: f.write(x + "\n"))
        self.log_artifact(str(summary_path), artifact_path="architecture")

    def end_run(self, status: str = "FINISHED"):
        """Ends the active run."""
        if self.active_run:
            run_id = self.active_run.info.run_id
            mlflow.end_run(status=status)
            print(f"[MLflow] ✓ Completed Run ID: {run_id} (Status: {status})")
            self.active_run = None

# Instantiate tracker
mlflow_tracker = DGNetNotebookMLflowTracker(experiment_name=EXPERIMENT_NAME)
mlflow.end_run()
# Start active run
run_name = f"DGNet-{MODEL_VERSION}-{BACKBONE}"
mlflow_tracker.start_run(
    run_name=run_name,
    tags={
        "model_version": MODEL_VERSION,
        "backbone": BACKBONE,
        "gradient_kernel": GRADIENT_KERNEL,
    },
    description=f"DGNet tactical training and COD evaluation run ({BACKBONE})."
)

# Log hyperparameters
mlflow_tracker.log_params({
    "model_name": "DGNet",
    "model_version": MODEL_VERSION,
    "context_backbone": BACKBONE,
    "gradient_kernel": GRADIENT_KERNEL,
    "image_size": f"[{IMG_SIZE[0]}, {IMG_SIZE[1]}]",
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "optimizer": "AdamW",
    "loss_function": "DGNetLoss (StructureLoss + MSE)",
    "train_samples": len(train_imgs),
    "test_samples": len(test_imgs),
    "total_parameters": total_params,
    "trainable_parameters": trainable_params
})

# Log configuration files
config_candidates = [
    PROJECT_ROOT / "params.yaml",
    PROJECT_ROOT / "configs/optimization/train.yaml",
    PROJECT_ROOT / "configs/data/data.yaml",
    PROJECT_ROOT / "configs/model/dgnet.yaml"
]
mlflow_tracker.log_configs([cp for cp in config_candidates if cp.exists()])

# Log model architecture summary
mlflow_tracker.log_model_summary(model)


[MLflow] Connected to database: sqlite:///C:/Users/Acer/Desktop/COD_PROJECT/metrics/mlflow_db/mlflow.db
[MLflow] Active Experiment:     'COD-Camouflage-Detection'
[MLflow] ✓ Started Run ID: 6af8b2b5d6ed4ad58b5d70ad1b3bfcba (Name: 'DGNet-v2.0_official_dgnet-mobilenet_v3_large')


## 6. Loss Function, Optimizer & Deep Supervision Graph
Compiles weighted BCE + weighted IoU + edge BCE loss with gradient clipping.

In [39]:
loss_fn = DGNetLoss(lambda_grad_loss=1.0, deep_supervision=True)
optimizer = tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=1e-4)

@tf.function
def train_step(x_batch, y_mask, y_grad):
    with tf.GradientTape() as tape:
        preds = model(x_batch, training=True, return_dict=True)
        loss = loss_fn(y_mask, y_grad, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

print("✓ Training step graph compiled.")

TypeError: Loss.__init__() got an unexpected keyword argument 'deep_supervision'

## 7. Model Training Loop with Live MLflow Tracking
Executes training with **live batch-level and epoch-level metric logging in MLflow**:
- `batch_train_loss` & `running_avg_train_loss` logged every 10 batches to MLflow.
- `epoch_train_loss`, `learning_rate`, and `epoch_time_seconds` logged at epoch boundaries.
- Best and latest `.weights.h5` checkpoints logged as MLflow artifacts.

In [37]:
RUN_TRAINING = False
TRAIN_EPOCHS = 5

if RUN_TRAINING:
    train_dataset = create_tf_dataset(train_imgs, train_masks, image_size=IMG_SIZE, batch_size=BATCH_SIZE, is_train=True)
    total_batches = len(train_imgs) // BATCH_SIZE
    best_loss = float("inf")
    
    for epoch in range(1, TRAIN_EPOCHS + 1):
        epoch_loss = 0.0
        batches = 0
        t0 = time.time()
        
        for step, (x_b, y_m, y_g) in enumerate(train_dataset):
            loss = train_step(x_b, y_m, y_g)
            loss_val = float(loss)
            epoch_loss += loss_val
            batches += 1
            
            # Live batch-level MLflow metric logging every 10 batches
            if batches % 10 == 0:
                global_step = (epoch - 1) * total_batches + batches
                mlflow_tracker.log_metrics({
                    "batch_train_loss": loss_val,
                    "running_avg_train_loss": epoch_loss / batches
                }, step=global_step)
                
            if batches % 25 == 0 or batches == total_batches:
                print(f"Epoch [{epoch:02d}/{TRAIN_EPOCHS:02d}] Batch [{batches:03d}/{total_batches:03d}] | Loss: {loss_val:.4f} | Avg: {epoch_loss/batches:.4f}")
                
        avg_loss = epoch_loss / max(batches, 1)
        elapsed = time.time() - t0
        print(f"--> Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s")
        
        # Log epoch-level metrics to MLflow
        mlflow_tracker.log_metrics({
            "epoch_train_loss": avg_loss,
            "epoch_time_seconds": elapsed,
            "learning_rate": LEARNING_RATE
        }, step=epoch)
        
        # Save epoch checkpoint and log to MLflow
        ckpt_path = CHECKPOINT_DIR / f"dgnet_{MODEL_VERSION}_epoch_{epoch:03d}.weights.h5"
        model.save_weights(str(ckpt_path))
        mlflow_tracker.log_artifact(str(ckpt_path), artifact_path="checkpoints")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_ckpt = CHECKPOINT_DIR / f"dgnet_{MODEL_VERSION}_best.weights.h5"
            model.save_weights(str(best_ckpt))
            mlflow_tracker.log_artifact(str(best_ckpt), artifact_path="checkpoints")
            print(f"  ★ Best model saved: {best_ckpt.name}")
else:
    print("ℹ Training loop skipped (set RUN_TRAINING = True to execute training).")

ℹ Training loop skipped (set RUN_TRAINING = True to execute training).


## 8. Benchmark Evaluation on Testing Split & MLflow Metrics Logging
Evaluates checkpoint performance on the test split, calculates official COD benchmark metrics, and logs them directly to the active MLflow run:
- `eval_pixel_accuracy`, `eval_mae`, `eval_iou`, `eval_dice`, `eval_f_measure` ($F_{0.3}$), `eval_precision`, `eval_recall`, `eval_specificity`, `eval_fps`.

In [38]:
# Search for best available weights
candidate_checkpoints = sorted(CHECKPOINT_DIR.glob("*.weights.h5"), reverse=True)
if candidate_checkpoints:
    best_ckpt = candidate_checkpoints[0]
    _ = model(dummy_input, training=False)
    model.load_weights(str(best_ckpt))
    ckpt_stem = best_ckpt.name.replace(".weights.h5", "").replace(".h5", "")
    print(f"✓ Loaded weights checkpoint: {best_ckpt.name}")
else:
    ckpt_stem = f"dgnet_{MODEL_VERSION}"
    print("ℹ No saved weights found; evaluating with initialized model weights.")

test_dataset = create_tf_dataset(test_imgs, test_masks, image_size=IMG_SIZE, batch_size=4, is_train=False)
metrics_acc = CODMetricsAccumulator(threshold=0.5, beta_sq=0.3)
t_start = time.perf_counter()

for x_b, y_m, _ in test_dataset:
    preds = model(x_b, training=False)
    metrics_acc.update(y_m, preds)

total_eval_time = time.perf_counter() - t_start
eval_results = metrics_acc.result()
fps = eval_results["sample_count"] / total_eval_time if total_eval_time > 0 else 0.0

# Log evaluation metrics to MLflow
mlflow_tracker.log_metrics({
    "eval_pixel_accuracy": float(eval_results["pixel_accuracy"]),
    "eval_mae": float(eval_results["mae"]),
    "eval_iou": float(eval_results["iou"]),
    "eval_dice": float(eval_results["dice"]),
    "eval_f_measure": float(eval_results["f_measure"]),
    "eval_precision": float(eval_results["precision"]),
    "eval_recall": float(eval_results["recall"]),
    "eval_specificity": float(eval_results["specificity"]),
    "eval_fps": float(fps)
})

print("\n" + "=" * 65)
print("           DGNet Benchmark Evaluation Results")
print("=" * 65)
print(f" Model Evaluated:         {ckpt_stem}")
print(f" Samples Evaluated:       {eval_results['sample_count']}")
print("-" * 65)
print(f" Pixel Accuracy:          {eval_results['pixel_accuracy']*100:.2f}%")
print(f" Mean Absolute Error:     {eval_results['mae']:.4f}")
print(f" IoU (Jaccard Index):     {eval_results['iou']:.4f}")
print(f" Dice Coefficient (F1):   {eval_results['dice']:.4f}")
print(f" F-measure (Fβ=0.3):     {eval_results['f_measure']:.4f}")
print("-" * 65)
print(f" Precision:               {eval_results['precision']:.4f}")
print(f" Recall:                  {eval_results['recall']:.4f}")
print(f" Specificity:             {eval_results['specificity']:.4f}")
print(f" Realtime Speed:          {fps:.1f} FPS")
print("=" * 65)
print("✓ Logged all evaluation metrics to active MLflow run.")

✓ Loaded weights checkpoint: dgnet_v2.0_official_dgnet_latest.weights.h5


KeyboardInterrupt: 

## 9. Clean Black-and-White Visual Comparison & MLflow Artifact Logging
Generates pure **black-and-white binary masks** (no color heatmaps or overlays):
- `{stem}_gt.png`: Ground Truth (0 = background, 255 = target)
- `{stem}_{model}_pred.png`: Model prediction (0 and 255)
- `{stem}_{model}_comparison.png`: Side-by-side `[ GROUND TRUTH | MODEL: {name} ]`
The comparison images are automatically registered as **MLflow run artifacts** under `prediction_comparisons/`.

In [ ]:
def generate_and_save_clean_visual_comparison(sample_idx=0, save_dir=PRED_DIR, model_tag=ckpt_stem):
    img_path = Path(test_imgs[sample_idx])
    gt_path = Path(test_masks[sample_idx])
    stem = img_path.stem
    
    img_raw = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_float = (img_resized.astype(np.float32) / 255.0 - IMAGENET_MEAN.numpy()) / IMAGENET_STD.numpy()
    
    gt_raw = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
    gt_bin = (cv2.resize(gt_raw, IMG_SIZE, interpolation=cv2.INTER_NEAREST) >= 128).astype(np.uint8) * 255
    
    inp_tensor = tf.expand_dims(tf.convert_to_tensor(img_float, dtype=tf.float32), 0)
    pred_prob = model(inp_tensor, training=False).numpy()[0, :, :, 0]
    pred_bin = (pred_prob >= 0.5).astype(np.uint8) * 255
    
    # Side-by-side pure black-and-white comparison panel
    h, w = IMG_SIZE
    sep = np.full((h, 4), 128, dtype=np.uint8)
    body = np.hstack([gt_bin, sep, pred_bin])
    
    header = np.zeros((36, body.shape[1]), dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(header, "GROUND TRUTH", (15, 24), font, 0.5, 255, 1, cv2.LINE_AA)
    cv2.putText(header, f"MODEL: {model_tag}", (w + 19, 24), font, 0.5, 255, 1, cv2.LINE_AA)
    h_div = np.full((2, body.shape[1]), 128, dtype=np.uint8)
    comparison_img = np.vstack([header, h_div, body])
    
    gt_file = save_dir / f"{stem}_gt.png"
    pred_file = save_dir / f"{stem}_{model_tag}_pred.png"
    comp_file = save_dir / f"{stem}_{model_tag}_comparison.png"
    
    cv2.imwrite(str(gt_file), gt_bin)
    cv2.imwrite(str(pred_file), pred_bin)
    cv2.imwrite(str(comp_file), comparison_img)
    
    # Log comparison image to active MLflow run
    mlflow_tracker.log_artifact(str(comp_file), artifact_path="prediction_comparisons")
    
    # Strictly display Ground Truth and Model Prediction masks (no colorized overlays)
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(gt_bin, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Ground Truth Mask", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(pred_bin, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"Prediction: {model_tag}", fontsize=12)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Saved clean GT:         {gt_file.name}")
    print(f"✓ Saved model prediction: {pred_file.name}")
    print(f"✓ Saved B&W comparison:   {comp_file.name} (Logged to MLflow)")

for idx in range(min(3, len(test_imgs))):
    generate_and_save_clean_visual_comparison(sample_idx=idx)

## 10. Edge Quantization (TFLite INT8) & MLflow Run Completion
Converts DGNet to TensorFlow Lite, saves strictly to `models/quantized/`, logs the `.tflite` model artifact to MLflow, and marks the MLflow run as `FINISHED`.

In [ ]:
TFLITE_EXPORT_PATH = PROJECT_ROOT / "models/quantized/dgnet_s_int8.tflite"
TFLITE_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Exporting DGNet to TFLite: {TFLITE_EXPORT_PATH.resolve()}...")

concrete_func = tf.function(lambda x: model(x, training=False)).get_concrete_function(
    tf.TensorSpec([1, IMG_SIZE[0], IMG_SIZE[1], 3], tf.float32)
)

converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model_bytes = converter.convert()

with open(TFLITE_EXPORT_PATH, "wb") as f:
    f.write(tflite_model_bytes)

size_mb = TFLITE_EXPORT_PATH.stat().st_size / (1024 * 1024)
print(f"✓ SUCCESS: TFLite model exported successfully ({size_mb:.2f} MB).")

# Log quantized model artifact to MLflow
mlflow_tracker.log_artifact(str(TFLITE_EXPORT_PATH), artifact_path="quantized_models")

# End the active MLflow run cleanly
mlflow_tracker.end_run(status="FINISHED")
print(f"✓ MLflow Run marked as FINISHED.")

## 11. MLflow Tracking Database & UI Inspection
Queries the local SQLite database (`metrics/mlflow_db/mlflow.db`) directly using `mlflow.search_runs()` and displays the logged experiments history.

In [ ]:
# Query runs from the MLflow SQLite database
mlflow.set_tracking_uri(mlflow_tracker.tracking_uri)
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

print(f"Total Runs Logged in '{EXPERIMENT_NAME}': {len(runs_df)}")
if not runs_df.empty:
    cols_to_show = [
        c for c in [
            "run_id", "tags.mlflow.runName", "status", "metrics.eval_iou",
            "metrics.eval_dice", "metrics.eval_pixel_accuracy", "metrics.eval_fps", "start_time"
        ] if c in runs_df.columns
    ]
    print("\nRecent MLflow Experiment Runs:")
    try:
        import pandas as pd
        pd.set_option('display.max_columns', 15)
        pd.set_option('display.width', 1000)
        print(runs_df[cols_to_show].head(5))
    except Exception:
        for idx, r in runs_df.head(5).iterrows():
            print(f"  • Run: {r.get('tags.mlflow.runName', r['run_id'])} | Status: {r['status']}")

print("\n" + "=" * 70)
print("                 To Launch the MLflow Web UI Locally:")
print("=" * 70)
print(f"Run the following command in PowerShell / Terminal:")
print(f"  mlflow ui --backend-store-uri {mlflow_tracker.tracking_uri} --port 5000")
print("Then open your browser to: http://localhost:5000")
print("=" * 70)